## Installing Dependacies

In [2]:
!pip install rouge-score
!pip install accelerate -U
!pip install gradio

In [17]:
from sklearn.model_selection import train_test_split

In [35]:
pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 54.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [3]:
overwrite_output_dir=True

### **Note**: After this restart the runtime

# **Cleaning Indian Food Recipe Dataset**



In [4]:
# ===============================
# 1. Imports
# ===============================
import pandas as pd
from datasets import load_dataset
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)

In [5]:
# Load the Indian Food dataset
from google.colab import drive
drive.mount('/content/drive')
data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/GEN AI/S2/IndianFoodDataset.csv")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
data.head()

,Srno,RecipeName,TranslatedRecipeName,Ingredients,TranslatedIngredients,PrepTimeInMins,CookTimeInMins,TotalTimeInMins,Servings,Cuisine,Course,Diet,Instructions,TranslatedInstructions,URL
0,1,Masala Karela Recipe,Masala Karela Recipe,"6 Karela (Bitter Gourd/ Pavakkai) - deseeded,S...","6 Karela (Bitter Gourd/ Pavakkai) - deseeded,S...",15,30,45,6,Indian,Side Dish,Diabetic Friendly,"To begin making the Masala Karela Recipe,de-se...","To begin making the Masala Karela Recipe,de-se...",https://www.archanaskitchen.com/masala-karela-...
1,2,टमाटर पुलियोगरे रेसिपी - Spicy Tomato Rice (Re...,Spicy Tomato Rice (Recipe),"2-1/2 कप चावल - पका ले,3 टमाटर,3 छोटा चमच्च बी...","2-1 / 2 cups rice - cooked, 3 tomatoes, 3 teas...",5,10,15,3,South Indian Recipes,Main Course,Vegetarian,टमाटर पुलियोगरे बनाने के लिए सबसे पहले टमाटर क...,"To make tomato puliogere, first cut the tomato...",http://www.archanaskitchen.com/spicy-tomato-ri...
2,3,Ragi Semiya Upma Recipe - Ragi Millet Vermicel...,Ragi Semiya Upma Recipe - Ragi Millet Vermicel...,"1-1/2 cups Rice Vermicelli Noodles (Thin),1 On...","1-1/2 cups Rice Vermicelli Noodles (Thin),1 On...",20,30,50,4,South Indian Recipes,South Indian Breakfast,High Protein Vegetarian,"To begin making the Ragi Vermicelli Recipe, fi...","To begin making the Ragi Vermicelli Recipe, fi...",http://www.archanaskitchen.com/ragi-vermicelli...
3,4,Gongura Chicken Curry Recipe - Andhra Style Go...,Gongura Chicken Curry Recipe - Andhra Style Go...,"500 grams Chicken,2 Onion - chopped,1 Tomato -...","500 grams Chicken,2 Onion - chopped,1 Tomato -...",15,30,45,4,Andhra,Lunch,Non Vegeterian,To begin making Gongura Chicken Curry Recipe f...,To begin making Gongura Chicken Curry Recipe f...,http://www.archanaskitchen.com/gongura-chicken...
4,5,आंध्रा स्टाइल आलम पचड़ी रेसिपी - Adrak Chutney ...,Andhra Style Alam Pachadi Recipe - Adrak Chutn...,"1 बड़ा चमच्च चना दाल,1 बड़ा चमच्च सफ़ेद उरद दाल,2...","1 tablespoon chana dal, 1 tablespoon white ura...",10,20,30,4,Andhra,South Indian Breakfast,Vegetarian,आंध्रा स्टाइल आलम पचड़ी बनाने के लिए सबसे पहले ...,"To make Andhra Style Alam Pachadi, first heat ...",https://www.archanaskitchen.com/andhra-style-a...


In [7]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1186 entries, 0 to 1185
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Srno                    1186 non-null   int64 
 1   RecipeName              1186 non-null   object
 2   TranslatedRecipeName    1186 non-null   object
 3   Ingredients             1185 non-null   object
 4   TranslatedIngredients   1185 non-null   object
 5   PrepTimeInMins          1186 non-null   int64 
 6   CookTimeInMins          1186 non-null   int64 
 7   TotalTimeInMins         1186 non-null   int64 
 8   Servings                1186 non-null   int64 
 9   Cuisine                 1186 non-null   object
 10  Course                  1186 non-null   object
 11  Diet                    1186 non-null   object
 12  Instructions            1186 non-null   object
 13  TranslatedInstructions  1186 non-null   object
 14  URL                     1186 non-null   object
dtypes: i

In [8]:
# Drop specified columns
data = data.drop(['Srno', 'RecipeName', 'Ingredients', 'Instructions'], axis=1)

In [9]:
data.head()

,TranslatedRecipeName,TranslatedIngredients,PrepTimeInMins,CookTimeInMins,TotalTimeInMins,Servings,Cuisine,Course,Diet,TranslatedInstructions,URL
0,Masala Karela Recipe,"6 Karela (Bitter Gourd/ Pavakkai) - deseeded,S...",15,30,45,6,Indian,Side Dish,Diabetic Friendly,"To begin making the Masala Karela Recipe,de-se...",https://www.archanaskitchen.com/masala-karela-...
1,Spicy Tomato Rice (Recipe),"2-1 / 2 cups rice - cooked, 3 tomatoes, 3 teas...",5,10,15,3,South Indian Recipes,Main Course,Vegetarian,"To make tomato puliogere, first cut the tomato...",http://www.archanaskitchen.com/spicy-tomato-ri...
2,Ragi Semiya Upma Recipe - Ragi Millet Vermicel...,"1-1/2 cups Rice Vermicelli Noodles (Thin),1 On...",20,30,50,4,South Indian Recipes,South Indian Breakfast,High Protein Vegetarian,"To begin making the Ragi Vermicelli Recipe, fi...",http://www.archanaskitchen.com/ragi-vermicelli...
3,Gongura Chicken Curry Recipe - Andhra Style Go...,"500 grams Chicken,2 Onion - chopped,1 Tomato -...",15,30,45,4,Andhra,Lunch,Non Vegeterian,To begin making Gongura Chicken Curry Recipe f...,http://www.archanaskitchen.com/gongura-chicken...
4,Andhra Style Alam Pachadi Recipe - Adrak Chutn...,"1 tablespoon chana dal, 1 tablespoon white ura...",10,20,30,4,Andhra,South Indian Breakfast,Vegetarian,"To make Andhra Style Alam Pachadi, first heat ...",https://www.archanaskitchen.com/andhra-style-a...


In [10]:
data.isnull().sum()

,0
TranslatedRecipeName,0
TranslatedIngredients,1
PrepTimeInMins,0
CookTimeInMins,0
TotalTimeInMins,0
Servings,0
Cuisine,0
Course,0
Diet,0
TranslatedInstructions,0


In [11]:
# Drop rows with missing values
data = data.dropna()

In [12]:
import re

In [13]:
def clean_text(text):
    # Remove non-alphanumeric characters and extra whitespaces
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Apply text cleaning to relevant columns
text_columns = ['TranslatedRecipeName', 'TranslatedIngredients', 'TranslatedInstructions']
for column in text_columns:
    data[column] = data[column].apply(clean_text)

In [14]:
# Save the cleaned dataset
data.to_csv("cleaned_IndianFoodDataset.csv", index=False)

# **Training GPT2 Model on cleaned_IndianFoodDataset**

In [15]:
# Load the cleaned dataset

data = pd.read_csv("cleaned_IndianFoodDataset.csv")

In [18]:
# Split the dataset into train, validation, and test sets

train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)
train_data, validation_data = train_test_split(train_data, test_size=0.1, random_state=42)

In [19]:
# ===============================
# 2. Load Dataset
# ===============================
data_files = {
    "train": "train_dataset.csv",
    "validation": "validation_dataset.csv",
    "test": "test_dataset.csv"
}

dataset = load_dataset("csv", data_files=data_files)

In [20]:
# ===============================
# 3. Initialize Tokenizer & Model (FIXED PAD TOKEN ERROR)
# ===============================
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

# 🔥 FIX: GPT-2 has no pad token → assign EOS as PAD
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [21]:
# ===============================
# 4. Clean Text Function
# ===============================
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


In [22]:
# ===============================
# 5. Tokenization Function (FIXED)
# ===============================
def tokenize_function(example):
    text = " ".join([
        clean_text(example.get("TranslatedRecipeName", "")),
        clean_text(example.get("TranslatedIngredients", "")),
        clean_text(example.get("TranslatedInstructions", ""))
    ])

    return tokenizer(
        text,
        truncation=True,
        padding="max_length",   # now safe ✅
        max_length=128
    )

In [23]:
# ===============================
# 6. Apply Tokenization
# ===============================
tokenized_datasets = dataset.map(
    tokenize_function,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/853 [00:00<?, ? examples/s]

Map:   0%|          | 0/95 [00:00<?, ? examples/s]

Map:   0%|          | 0/237 [00:00<?, ? examples/s]

In [24]:
# ===============================
# 7. Data Collator
# ===============================
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [26]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=4
)

In [39]:
# ===============================
# 9. Trainer
# ===============================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],  # important
    data_collator=data_collator
)

In [40]:
# ===============================
# 10. Train Model
# ===============================
trainer.train()

# ===============================
# 11. Save Model
# ===============================
model.save_pretrained("./gpt2-recipe-model")
tokenizer.save_pretrained("./gpt2-recipe-model")

Step,Training Loss
500,1.780948


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./gpt2-recipe-model/tokenizer_config.json',
 './gpt2-recipe-model/tokenizer.json')

In [31]:
import torch

def generate_recipe(prompt, max_length=100):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.to(device)  # ensure model is on device

    inputs = tokenizer(prompt, return_tensors="pt", padding=True)
    inputs = {key: val.to(device) for key, val in inputs.items()}  # 🔥 FIX

    outputs = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=max_length,
        do_sample=True,              # 🔥 REQUIRED for temperature/top_p
        top_k=50,
        top_p=0.95,
        temperature=0.7,
        no_repeat_ngram_size=2,
        pad_token_id=tokenizer.pad_token_id
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [33]:
# ===============================
# 13. Test Generation
# ===============================
print(generate_recipe("To make butter Chicken Curry"))

To make butter Chicken Curry Recipe 1 cup Butter34 cup Milk14 cup Fresh coconut milk14 teaspoon SugarSalt to taste1 teaspoon Curd Dahi Yogurt14 tablespoon Coriander Dhania Seeds12 cup Coraichi PodsSeeds chopped12 teaspoon Amchur Dry Mango PowderSalt and Pepper for garnish To prepare the Chicken Curry Recipe keep asideHeat a pressure cooker with a little oil Add butter in it Add the eggs and milk and cook for about 15 minutesAdd the milk to


In [34]:
# Save the model

trainer.save_model("gpt2-indian-food")

# Save the configuration file
model = trainer.model
model.config.save_pretrained("gpt2-indian-food_config.json")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [42]:
results = trainer.evaluate(tokenized_datasets["test"])
print(results)

Step,Training Loss,Validation Loss
63,No log,2.186675


{'eval_loss': 2.1866750717163086}


# **Model Evalution - BLUE & ROUGE**

In [43]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

# Download NLTK resources
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [44]:
# Load the trained model and tokenizer
model = GPT2LMHeadModel.from_pretrained("gpt2-indian-food")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# Function to generate text using the model
def generate_text(prompt, max_length=100, temperature=1.0):
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    output = model.generate(input_ids, max_length=max_length, temperature=temperature, num_beams=5, no_repeat_ngram_size=2)
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    return generated_text

# Example prompts and references
prompts = ["To make Masala Karela,",
           "Tomato Puliogere is prepared by",
           "Ragi Vermicelli Recipe starts with",
           "Gongura Chicken Curry Recipe first prep all the ingredients",
           "Andhra Style Alam Pachadi is made by"]

references = ["To begin making the Masala Karela Recipe, de-seed the karela and slice.",
              "To make tomato puliogere, first cut the tomatoes.",
              "To begin making the Ragi Vermicelli Recipe, first steam the ragi vermicelli.",
              "To begin making Gongura Chicken Curry Recipe, first prep all the ingredients.",
              "To make Andhra Style Alam Pachadi, first heat oil in a pan."]

# Generate text and calculate BLEU and ROUGE scores
smooth = SmoothingFunction().method1  # Define smoothing function for BLEU
bleu_scores = []
rouge_scores = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

for prompt, reference in zip(prompts, references):
    generated_text = generate_text(prompt)

    # BLEU score
    bleu_score = sentence_bleu([reference.split()], generated_text.split(), smoothing_function=smooth)
    bleu_scores.append(bleu_score)
    print(f"Prompt: {prompt}\nGenerated: {generated_text}\nReference: {reference}\nBLEU Score: {bleu_score:.4f}\n")

    # ROUGE scores
    rouge_score = rouge_scores.score(generated_text, reference)
    print(f"ROUGE Scores: {rouge_score}\n")

# Average BLEU score
average_bleu = sum(bleu_scores) / len(bleu_scores)
print(f"Average BLEU Score: {average_bleu:.4f}")


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt: To make Masala Karela,
Generated: To make Masala Karela, we will first cook the masala in a heavy bottomed pan and add it to the pressure cooker and cook it for 2 whistles Once done add the lentils and saute till it becomes soft Now add turmeric powder red chilli powder salt as per taste and let it cook for 5 to 10 minutes After 5 minutes turn off the gas and keep it aside Now heat a pan on medium heat add mustard seeds green coriander leaves green chillies and tur
Reference: To begin making the Masala Karela Recipe, de-seed the karela and slice.
BLEU Score: 0.0031

ROUGE Scores: {'rouge1': Score(precision=0.6153846153846154, recall=0.09195402298850575, fmeasure=0.16), 'rouge2': Score(precision=0.16666666666666666, recall=0.023255813953488372, fmeasure=0.04081632653061224), 'rougeL': Score(precision=0.46153846153846156, recall=0.06896551724137931, fmeasure=0.11999999999999998)}



The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt: Tomato Puliogere is prepared by
Generated: Tomato Puliogere is prepared by mixing tomato puree with water and salt in a mixer grinder until smooth and creamy. Keep aside To begin making the Tomato Puliiere recipe firstly heat a heavy bottomed pan on medium heat add the tomatoes and saute until they turn golden brown and crisp Turn off the heat and let it rest for 10 minutes Add the chopped tomatoes to the blender and blend until you get a smooth mixture Keep it aside In a large mixing bowl combine the tomato
Reference: To make tomato puliogere, first cut the tomatoes.
BLEU Score: 0.0027

ROUGE Scores: {'rouge1': Score(precision=0.75, recall=0.06896551724137931, fmeasure=0.12631578947368421), 'rouge2': Score(precision=0.2857142857142857, recall=0.023255813953488372, fmeasure=0.04301075268817204), 'rougeL': Score(precision=0.625, recall=0.05747126436781609, fmeasure=0.10526315789473682)}



The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt: Ragi Vermicelli Recipe starts with
Generated: Ragi Vermicelli Recipe starts with 2 cups of rice and 2 tablespoons of water1 teaspoon Turmeric powder Haldi12 teaspoon Red Chilli powder1 tablespoon Coriander Dhania Leaves finely choppedSalt to tasteSunflower Oil for deep frying To begin making the Arhar Dahi Yogurt Recipe wash and soak the rice in water for at least 2 hoursHeat oil in a heavy bottomed pan and add the curd and saute till it becomes soft and fluffyAdd the chopped cori
Reference: To begin making the Ragi Vermicelli Recipe, first steam the ragi vermicelli.
BLEU Score: 0.0372

ROUGE Scores: {'rouge1': Score(precision=0.6666666666666666, recall=0.10666666666666667, fmeasure=0.1839080459770115), 'rouge2': Score(precision=0.45454545454545453, recall=0.06756756756756757, fmeasure=0.11764705882352941), 'rougeL': Score(precision=0.5, recall=0.08, fmeasure=0.13793103448275865)}



The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt: Gongura Chicken Curry Recipe first prep all the ingredients
Generated: Gongura Chicken Curry Recipe first prep all the ingredients and keep it readyHeat oil in a heavy bottomed pan Add onion and garlic and saute for 2 to 3 minutesAdd tomato puree and cook till the tomatoes turn golden brownAdd salt and pepper to taste and let it cook for 5 to 10 minutesOnce it is done add chicken to the pan and roast it for 10 to 15 minutesHeat a pan on medium heat add mustard seeds turmeric powder coriander powder red chilli powder and
Reference: To begin making Gongura Chicken Curry Recipe, first prep all the ingredients.
BLEU Score: 0.0379

ROUGE Scores: {'rouge1': Score(precision=0.8333333333333334, recall=0.11627906976744186, fmeasure=0.20408163265306123), 'rouge2': Score(precision=0.7272727272727273, recall=0.09411764705882353, fmeasure=0.16666666666666666), 'rougeL': Score(precision=0.75, recall=0.10465116279069768, fmeasure=0.1836734693877551)}

Prompt: Andhra Style Alam Pachadi is made

# **Checking Model performace with Gradio Interface**

In [45]:
import gradio as gr
from transformers import GPT2LMHeadModel, GPT2Tokenizer

In [46]:
# Load the GPT-2 tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2-indian-food")  # Use the path where your model is saved

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [47]:
def generate_text(prompt):
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    output = model.generate(input_ids, max_length=50, num_beams=5, no_repeat_ngram_size=2, top_k=50, top_p=0.95, temperature=0.7)
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    return generated_text


In [48]:
# Create a Gradio Interface
iface = gr.Interface(
    fn=generate_text,
    inputs="text",
    outputs="text",
    live=True,
    examples=[["To make a delicious curry, ", "you need to start with fresh vegetables and aromatic spices."]]
)

In [49]:
iface.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1fa983613b83cc16ad.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
